# 06 — Fixed Income: TRACE & FISD

OTC bond trade data, bid-ask spreads, and bond characteristics.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os
from dotenv import load_dotenv

# Load config/.env if present (repo keeps credentials there), else a local .env.
load_dotenv("config/.env")
load_dotenv()


def _env(*names, default=None):
    """First non-empty env var among names (accepts canonical + alias names)."""
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return default


ENDPOINT = _env("R2_ENDPOINT")
KEY_ID   = _env("R2_ACCESS_KEY_ID", "R2_KEY_ID")
SECRET   = _env("R2_SECRET_ACCESS_KEY", "R2_SECRET")
BUCKET   = _env("R2_BUCKET", default="quantt-historical-market-data")
assert ENDPOINT and KEY_ID and SECRET, "Set R2 credentials in config/.env."

con = duckdb.connect()
con.execute("""
    INSTALL httpfs; LOAD httpfs;
    SET s3_endpoint    = '{endpoint}';
    SET s3_access_key_id     = '{key_id}';
    SET s3_secret_access_key = '{secret}';
    SET s3_region = 'auto';
    SET s3_url_style = 'path';
""".format(
    endpoint = ENDPOINT.replace('https://', ''),
    key_id   = KEY_ID,
    secret   = SECRET,
))


def r2(schema, table):
    """Return the S3 path for a given schema/table."""
    return f"s3://{BUCKET}/wrds/{schema}/{table}.parquet"

def q(sql):
    return con.execute(sql).df()

print("Connected to R2.")

## Bond Characteristics: FISD

In [ ]:
fisd = q(f"""
    SELECT f.issue_id, f.cusip_id, f.issuer_id, f.coupon, f.maturity,
           f.offering_amt, f.bond_type, f.callable, f.putable,
           f.rating_moody, f.rating_sp
    FROM read_parquet('{r2('fisdsamp_all', 'fisd_mergedissue_samp')}') f
    WHERE f.maturity >= '2025-01-01' LIMIT 10000
""")
print(f"{len(fisd):,} bonds")
fisd.head(5)

## Coupon Distribution by Bond Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for btype in ['CDEB', 'CMTN', 'CZ', 'CS']:
    sub = fisd[fisd['bond_type'] == btype]['coupon'].dropna()
    if len(sub) > 10:
        ax.hist(sub, bins=40, alpha=0.5, label=btype, edgecolor='none')
ax.set_xlabel('Coupon Rate (%)'); ax.set_title('Coupon Distribution by Bond Type (FISD)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## TRACE Enhanced Bond Trades

In [ ]:
trace = q(f"""
    SELECT trd_exctn_dt, cusip_id, rptd_pr, entrd_vol_qt, yld_pt, trc_st
    FROM read_parquet('{r2('trace_enhanced', 'trace_enhanced')}')
    WHERE trd_exctn_dt >= '2023-01-01' AND trc_st = 'T' LIMIT 100000
""")
print(f"{len(trace):,} trades | {trace.cusip_id.nunique():,} unique bonds")
trace.head(5)

## Roll (1984) Effective Spread

In [ ]:
import numpy as np
daily_prices = (
    trace.groupby(['cusip_id', 'trd_exctn_dt'])['rptd_pr']
    .median().reset_index().sort_values(['cusip_id', 'trd_exctn_dt'])
)
daily_prices['ret'] = daily_prices.groupby('cusip_id')['rptd_pr'].pct_change()
def roll_spread(x):
    if len(x) < 20: return np.nan
    cov = x['ret'].cov(x['ret'].shift(1))
    return 2 * np.sqrt(max(-cov, 0))
spreads = daily_prices.groupby('cusip_id').apply(roll_spread).dropna()
print(f"Median Roll spread: {spreads.median():.4f}")